In [17]:
import numpy as np
import pickle
class AdamOptimizer:
    def __init__(self, learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-7):
        self.learning_rate = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = None
        self.v = None
        self.t = 0

    def update(self, param, grad):
        if self.m is None:
            self.m = np.zeros_like(grad)
            self.v = np.zeros_like(grad)
        
        self.t += 1
        self.m = self.beta1 * self.m + (1 - self.beta1) * grad
        self.v = self.beta2 * self.v + (1 - self.beta2) * (grad ** 2)
        
        m_hat = self.m / (1 - self.beta1 ** self.t)
        v_hat = self.v / (1 - self.beta2 ** self.t)
        
        param -= self.learning_rate * m_hat / (np.sqrt(v_hat) + self.epsilon)
        return param

class DenseLayer:
    def __init__(self, input_size, output_size, learning_rate=0.001):
        self.weights = np.random.randn(input_size, output_size) * 0.01
        self.biases = np.zeros((1, output_size))
        self.optimizer_w = AdamOptimizer(learning_rate)
        self.optimizer_b = AdamOptimizer(learning_rate)
    
    def forward(self, inputs, training=True):
        self.inputs = inputs
        return np.dot(inputs, self.weights) + self.biases
    
    def backward(self, d_output):
        d_weights = np.dot(self.inputs.T, d_output)
        d_biases = np.sum(d_output, axis=0, keepdims=True)
        d_inputs = np.dot(d_output, self.weights.T)
        
        # Updating weights and biases with Adam optimizer
        self.weights = self.optimizer_w.update(self.weights, d_weights)
        self.biases = self.optimizer_b.update(self.biases, d_biases)
        
        return d_inputs

class BatchNormalization:
    def __init__(self, size, epsilon=1e-7, momentum=0.9, learning_rate=0.001):
        self.epsilon = epsilon
        self.momentum = momentum
        self.gamma = np.ones((1, size))
        self.beta = np.zeros((1, size))
        self.optimizer_gamma = AdamOptimizer(learning_rate)
        self.optimizer_beta = AdamOptimizer(learning_rate)
        
        self.running_mean = np.zeros((1, size))
        self.running_variance = np.ones((1, size))
    
    def forward(self, inputs, training=True):
        self.inputs = inputs
        if training:
            batch_mean = np.mean(inputs, axis=0, keepdims=True)
            batch_variance = np.var(inputs, axis=0, keepdims=True)
            self.normalized = (inputs - batch_mean) / np.sqrt(batch_variance + self.epsilon)
            output = self.gamma * self.normalized + self.beta
            
            self.running_mean = self.momentum * self.running_mean + (1 - self.momentum) * batch_mean
            self.running_variance = self.momentum * self.running_variance + (1 - self.momentum) * batch_variance
            self.batch_mean = batch_mean
            self.batch_variance = batch_variance
            
            return output
        else:
            normalized = (inputs - self.running_mean) / np.sqrt(self.running_variance + self.epsilon)
            return self.gamma * normalized + self.beta
    
    def backward(self, d_output):
        batch_size = d_output.shape[0]
        
        d_beta = np.sum(d_output, axis=0, keepdims=True)
        d_gamma = np.sum(d_output * self.normalized, axis=0, keepdims=True)
        d_normalized = d_output * self.gamma
        d_variance = np.sum(d_normalized * (self.inputs - self.batch_mean) * -0.5 * np.power(self.batch_variance + self.epsilon, -1.5), axis=0, keepdims=True)
        d_mean = np.sum(d_normalized * -1 / np.sqrt(self.batch_variance + self.epsilon), axis=0, keepdims=True) + d_variance * np.sum(-2 * (self.inputs - self.batch_mean), axis=0) / batch_size
        d_inputs = d_normalized / np.sqrt(self.batch_variance + self.epsilon) + d_variance * 2 * (self.inputs - self.batch_mean) / batch_size + d_mean / batch_size
        
        # Updating gamma and beta with Adam optimizer
        self.gamma = self.optimizer_gamma.update(self.gamma, d_gamma)
        self.beta = self.optimizer_beta.update(self.beta, d_beta)
        
        return d_inputs

class ReLU:
    def forward(self, inputs,training=True):
        self.inputs = inputs
        return np.maximum(0, inputs)
    
    def backward(self, d_output):
        d_inputs = d_output.copy()
        d_inputs[self.inputs <= 0] = 0
        return d_inputs

class Dropout:
    def __init__(self, rate):
        self.rate = rate  # Dropout rate (e.g., 0.5 for 50% dropout,means half of the neurons are off)
        self.mask = None
    
    def forward(self, inputs, training=True):
        if training:
            # Creating a mask to randomly drop units with probability `rate`
            self.mask = np.random.rand(*inputs.shape) > self.rate
            # Scaling the remaining units to maintain expected output magnitude
            return inputs * self.mask / (1 - self.rate)
        else:
            # During inference, we just return the inputs unchanged
            return inputs
    
    def backward(self, d_output):
        # Gradient flow is blocked through dropped units
        return d_output * self.mask

class Softmax:
    def forward(self, inputs,training=True):
        exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        probabilities = exp_values / np.sum(exp_values, axis=1, keepdims=True)
        return probabilities
    # assuming softmax will be at last and loss function is cross entropy. using y_pred - y_true as d_output
    def backward(self, d_output):
        return d_output


In [18]:
from sklearn.metrics import f1_score

# class model to initialize the model with list of layers,followed by forward and backward pass
training_loss=[]
validation_loss=[]
training_accuracy=[]
validation_accuracy=[]
validation_f1_score=[]
class Model:
    def __init__(self, layers):
        self.layers = layers
    
    def forward(self, X, training=True):
        for layer in self.layers:
            X = layer.forward(X, training)
        return X
    
    def backward(self, d_output):
        for layer in reversed(self.layers):
            d_output = layer.backward(d_output)
    
    def one_hot_encode(self, y, num_classes):
        one_hot = np.zeros((len(y), num_classes))
        one_hot[np.arange(len(y)), y] = 1
        return one_hot
    
    def train(self, X, y, epochs, batch_size, num_classes, validation_data=None):
        y_one_hot = self.one_hot_encode(y, num_classes)
        for epoch in range(epochs):
            epoch_loss = 0
            epoch_accuracy = 0
            num_batches = 0
            
            for i in range(0, len(X), batch_size):
                X_batch = X[i:i+batch_size]
                y_batch = y_one_hot[i:i+batch_size]
                y_pred = self.forward(X_batch)
                
                loss = -np.sum(y_batch * np.log(y_pred + 1e-7)) / len(X_batch)
                accuracy = np.mean(np.argmax(y_batch, axis=1) == np.argmax(y_pred, axis=1))
                d_output = y_pred - y_batch
                self.backward(d_output)
                
                epoch_loss += loss
                epoch_accuracy += accuracy
                num_batches += 1
            
            epoch_loss /= num_batches
            epoch_accuracy /= num_batches
            
            print(f'Epoch {epoch+1}/{epochs} - loss: {epoch_loss} - accuracy: {epoch_accuracy}')
            training_loss.append(epoch_loss)
            training_accuracy.append(epoch_accuracy)
            
            # validation loss and f1 score
            if validation_data:
                X_val, y_val = validation_data
                y_val_one_hot = self.one_hot_encode(y_val, num_classes)
                y_val_pred = self.forward(X_val, training=False)
                val_loss = -np.sum(y_val_one_hot * np.log(y_val_pred + 1e-7)) / len(X_val)
                val_accuracy = np.mean(np.argmax(y_val_one_hot, axis=1) == np.argmax(y_val_pred, axis=1))
                val_f1_score = f1_score(y_val, np.argmax(y_val_pred, axis=1), average='weighted')
                print(f'Validation loss: {val_loss} - Validation accuracy: {val_accuracy} - Validation F1 score: {val_f1_score}')
                validation_loss.append(val_loss)
                validation_accuracy.append(val_accuracy)
                validation_f1_score.append(val_f1_score)
    
    def save_model(self, filename):
        model_data = []
        for layer in self.layers:
            if isinstance(layer, DenseLayer):
                model_data.append({
                    'type': 'DenseLayer',
                    'weights': layer.weights,
                    'biases': layer.biases
                })
            elif isinstance(layer, BatchNormalization):
                model_data.append({
                    'type': 'BatchNormalization',
                    'gamma': layer.gamma,
                    'beta': layer.beta,
                    'running_mean': layer.running_mean,
                    'running_variance': layer.running_variance
                })
            elif isinstance(layer, Dropout):
                model_data.append({
                    'type': 'Dropout',
                    'rate': layer.rate
                })
            elif isinstance(layer, ReLU):
                model_data.append({
                    'type': 'ReLU'
                })
            elif isinstance(layer, Softmax):
                model_data.append({
                    'type': 'Softmax'
                })
        
        with open(filename, 'wb') as f:
            pickle.dump(model_data, f)

    def load_model(self, filename):
        with open(filename, 'rb') as f:
            model_data = pickle.load(f)
        
        layers = []
        for layer_data in model_data:
            if layer_data['type'] == 'DenseLayer':
                layer = DenseLayer(0, 0)  # dummy initialization
                layer.weights = layer_data['weights']
                layer.biases = layer_data['biases']
            elif layer_data['type'] == 'BatchNormalization':
                layer = BatchNormalization(0)  # dummy initialization
                layer.gamma = layer_data['gamma']
                layer.beta = layer_data['beta']
                layer.running_mean = layer_data['running_mean']
                layer.running_variance = layer_data['running_variance']
            elif layer_data['type'] == 'Dropout':
                layer = Dropout(layer_data['rate'])
            elif layer_data['type'] == 'ReLU':
                layer = ReLU()
            elif layer_data['type'] == 'Softmax':
                layer = Softmax()
            layers.append(layer)
        
        self.layers = layers

In [19]:
from torchvision import datasets, transforms

# define the transformation
transform = transforms.ToTensor()

# download and load the training data
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)

# load test dataset separately
# test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
with open('a2.pkl', 'rb') as a2:
  test_dataset = pickle.load(a2)

In [20]:
import torch
#minmax scaling
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

# convert the image data to a numpy 1D array
X_train = train_dataset.data.numpy().reshape(-1, 28*28)/255
y_train = train_dataset.targets.numpy()

# train and validation split, using train_test_split from sklearn
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=123)

# convert the image data to a numpy 1D array for test dataset
# X_test = test_dataset.data.numpy().reshape(-1, 28*28)/255
# y_test = test_dataset.targets.numpy()
X_test = torch.stack([x for x, _ in test_dataset]).numpy().reshape(-1, 28*28)
y_test = torch.stack([y for _, y in test_dataset]).numpy()
scaler.fit_transform(X_test)
model_list = []


# def train_models_with_different_learning_rates(learning_rates):
#     for i, lr in enumerate(learning_rates):
#         print(f"Training iteration {i+1} with learning rate {lr}")
        
#         # Initialize the model
#         model = Model([
#             DenseLayer(28*28, 128, learning_rate=lr),
#             BatchNormalization(128, learning_rate=lr),
#             ReLU(),
#             Dropout(0.5),
#             DenseLayer(128, 64, learning_rate=lr),
#             BatchNormalization(64, learning_rate=lr),
#             ReLU(),
#             Dropout(0.5),
#             DenseLayer(64, 10, learning_rate=lr),
#             BatchNormalization(10, learning_rate=lr),
#             Softmax()
#         ])

#         # Initialize the second model with a different architecture
#         model2 = Model([
#             DenseLayer(28*28, 256, learning_rate=lr),
#             BatchNormalization(256, learning_rate=lr),
#             ReLU(),
#             Dropout(0.3),
#             DenseLayer(256, 128, learning_rate=lr),
#             BatchNormalization(128, learning_rate=lr),
#             ReLU(),
#             Dropout(0.3),
#             DenseLayer(128, 10, learning_rate=lr),
#             BatchNormalization(10, learning_rate=lr),
#             Softmax()
#         ])

#         # Initialize the third model with another different architecture
#         model3 = Model([
#             DenseLayer(28*28, 512, learning_rate=lr),
#             BatchNormalization(512, learning_rate=lr),
#             ReLU(),
#             Dropout(0.4),
#             DenseLayer(512, 256, learning_rate=lr),
#             BatchNormalization(256, learning_rate=lr),
#             ReLU(),
#             Dropout(0.4),
#             DenseLayer(256, 10, learning_rate=lr),
#             BatchNormalization(10, learning_rate=lr),
#             Softmax()
#         ])

#         # Train the models
#         model.train(X_train, y_train, epochs=10, batch_size=32, num_classes=10,validation_data=(X_val, y_val))
#         model2.train(X_train, y_train, epochs=10, batch_size=32, num_classes=10,validation_data=(X_val, y_val))
#         model3.train(X_train, y_train, epochs=10, batch_size=32, num_classes=10,validation_data=(X_val, y_val))
#         model_list.append(model)
#         model_list.append(model2)
#         model_list.append(model3)

# # Example usage
# learning_rates = [0.001, 0.01, 0.0001, 0.00001]
# train_models_with_different_learning_rates(learning_rates)

In [21]:

# import matplotlib.pyplot as plt

# # Plot training and validation loss
# for i in range(0, len(training_loss), 10):
#     plt.figure(figsize=(12, 6))
#     plt.subplot(1, 2, 1)
#     plt.plot(training_loss[i:i+10], label='Training Loss')
#     plt.plot(validation_loss[i:i+10], label='Validation Loss')
#     plt.xlabel('Epochs')
#     plt.ylabel('Loss')
#     plt.title(f'Model {i//10 + 1} Training and Validation Loss')
#     plt.legend()

#     # Plot training and validation accuracy
#     plt.subplot(1, 2, 2)
#     plt.plot(training_accuracy[i:i+10], label='Training Accuracy')
#     plt.plot(validation_accuracy[i:i+10], label='Validation Accuracy')
#     plt.xlabel('Epochs')
#     plt.ylabel('Accuracy')
#     plt.title(f'Model {i//10 + 1} Training and Validation Accuracy')
#     plt.legend()
#     plt.show()

# plt.show()

# # Plot validation F1 score
# # plt.figure(figsize=(6, 6))
# # plt.plot(validation_f1_score, label='Validation F1 Score')
# # plt.xlabel('Epochs')
# # plt.ylabel('F1 Score')
# # plt.title('Validation F1 Score')
# # plt.legend()
# for i in range(0, len(validation_f1_score), 10):
#     plt.figure(figsize=(6, 6))
#     plt.plot(validation_f1_score[i:i+10], label='Validation F1 Score')
#     plt.xlabel('Epochs')
#     plt.ylabel('F1 Score')
#     plt.title(f'Model {i//10 + 1} Validation F1 Score')
#     plt.legend()
#     plt.show()

# from sklearn.metrics import confusion_matrix
# import seaborn as sns

# # Run each model on validation dataset and create confusion matrix
# for i, model in enumerate(model_list):
#     y_val_pred = model.forward(X_val, training=False)
#     y_val_pred = np.argmax(y_val_pred, axis=1)
#     cm = confusion_matrix(y_val, y_val_pred)
    
#     plt.figure(figsize=(8, 6))
#     sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
#     plt.xlabel('Predicted')
#     plt.ylabel('True')
#     plt.title(f'Confusion Matrix for Model {i+1}')
#     plt.show()

# plt.show()


In [22]:
# Save the best model based on validation F1 score
# best_model_index = np.argmax(validation_f1_score)
# # convert the index into integer
# best_model_index=int(best_model_index/10)
# best_model = model_list[best_model_index]
# best_model.save_model('best_model.pkl')

# Load the previously saved model
loaded_model = Model([])
loaded_model.load_model('model_1905040.pkl')

# Test the loaded model
y_pred = loaded_model.forward(X_test, training=False)
y_pred = np.argmax(y_pred, axis=1)
accuracy = np.mean(y_pred == y_test)
print(f'Accuracy of the loaded model: {accuracy}')
# f1 score
f1 = f1_score(y_test, y_pred, average='weighted')
print(f'F1 score of the loaded model: {f1}')

Accuracy of the loaded model: 0.24120742176682358
F1 score of the loaded model: 0.10044385074805486
